In [7]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter, TextFormatter

import yt_dlp
import re
from pytube import Playlist

In [8]:
url_playlist = "https://www.youtube.com/watch?v=4-h4rZPvbBg&list=PL3T5QtWK2C4Mavmjvqrt8hYBCNI6Sic_9"

playlist = Playlist(url_playlist)


In [9]:
playlist._video_regex = re.compile(r"\"url\":\"(/watch\?v=[\w-]*)")

In [10]:
len(playlist.video_urls)


262

In [14]:
def get_video_urls(channel_url):
    ydl_opts = {
        'quiet': True,
        'extract_flat': True,
        'playlistend': 1000,  # Limit to first 1000 videos
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        
        if 'entries' in info:
            video_urls = [entry['url'] for entry in info['entries'] if 'url' in entry]
            return video_urls


def get_youtube_video_info(video_url):
    ydl_opts = {
        'noplaylist': True,
        'quiet': True,
        'no_warnings': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        video_info = ydl.extract_info(video_url, download=False)
        return {
            "video_id": video_info['id'],
            "title": video_info['title'],
            "upload_date": video_info['upload_date'],
            "channel": video_info['channel'],
            "duration": video_info['duration_string'],
            "caption": get_youtube_video_transcript(video_info['id'])
        }
    
def get_youtube_video_transcript(video_id):
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['ko'])
    return " ".join(line['text'] for line in transcript)

In [16]:
video_info = get_youtube_video_info(playlist.video_urls[0])
video_info

{'video_id': '4-h4rZPvbBg',
 'title': '비뇨기과 아재들이 쿨한 이유',
 'upload_date': '20250306',
 'channel': '동공이 약사',
 'duration': '28',
 'caption': '[음악] 아네 처방 좀 받았습니다음 해피 하나 해피 뭐가 해피요 아 아니요 아니요 요런 종류의 약을 해피드럭이라고 부르거든요 아 그래도 남자 생님이 편하긴 하네 그러세요 저희도 편해요 atc 넣을 것도 없이 제로 꺼내 드리면 돼서 응 그래요 카드로 하시나요 응 영수증 드릴까요 응 됐어 안녕히 가세요 [음악]'}

In [17]:
transcript = YouTubeTranscriptApi.get_transcript(video_info['video_id'], languages=['ko'])
" ".join(line['text'] for line in transcript)

'[음악] 아네 처방 좀 받았습니다음 해피 하나 해피 뭐가 해피요 아 아니요 아니요 요런 종류의 약을 해피드럭이라고 부르거든요 아 그래도 남자 생님이 편하긴 하네 그러세요 저희도 편해요 atc 넣을 것도 없이 제로 꺼내 드리면 돼서 응 그래요 카드로 하시나요 응 영수증 드릴까요 응 됐어 안녕히 가세요 [음악]'